#### FIFTH ATTEMPT

## Temporal Attack Prediction

### OpTC + Windows-APT datasets

**Overview:**

The combined dataset integrates OpTC endpoint telemetry with Windows-APT 2025 telemetry to provide a broader set of benign and attack-related system activities for temporal cyberattack prediction. The two datasets were harmonised into a common schema containing timestamp, event_action, event_object, protocol, process and thread identifiers (pid, ppid, tid), hostname, label, and dataset_source.
For the LSTM and GRU experiment, timestamps are used to chronologically organise events and construct temporal sequences, allowing the models to learn patterns in preceding system activity and predict whether malicious activity will occur within a subsequent time window.

**Project goal:**

The aim is temporal prediction rather than event-level detection: use activity from the previous 10 minutes to predict whether malicious activity will occur during the next 5 minutes. The raw timestamp is retained for chronological ordering and window construction. hour and minute are also retained as model features, consistent with the previous experiment. Windows are created separately within each dataset source, hostname and date so sequences do not cross hosts, datasets or day boundaries.



In [145]:
# Imports
%matplotlib inline

from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    GRU,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

seed = 7

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [146]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Combined Dataset

In [147]:
combination_path = Path(
    "/content/drive/MyDrive/solutions/Combination_OpTC_APT"
)

train_data = pd.read_parquet(
    combination_path / "combined_train.parquet"
)

test_data = pd.read_parquet(
    combination_path / "combined_test.parquet"
)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)



Train shape: (926673, 10)
Test shape: (36550, 10)


### Prepare the timestamp and feature datatypes (Same as previous experiment)

In [148]:
def convert_timestamp_by_source(df):

    result = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns, UTC]"
    )

    # Windows-APT
    apt_mask = df["dataset_source"].eq("Windows-APT")

    result.loc[apt_mask] = pd.to_datetime(
        df.loc[apt_mask, "timestamp"],
        format="%b %d, %Y @ %H:%M:%S.%f",
        errors="coerce",
        utc=True
    )

    # OpTC
    optc_mask = df["dataset_source"].eq("OpTC")

    optc_raw = df.loc[
        optc_mask,
        "timestamp"
    ]

    numeric = pd.to_numeric(
        optc_raw,
        errors="coerce"
    )

    numeric_mask = numeric.notna()

    # Numeric OpTC timestamps
    if numeric_mask.any():

        values = numeric[numeric_mask]

        median_value = (
            values.abs().median()
        )

        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"

        result.loc[
            values.index
        ] = pd.to_datetime(
            values,
            unit=unit,
            errors="coerce",
            utc=True
        )

    # String-formatted OpTC timestamps
    string_idx = optc_raw.index[
        ~numeric_mask
    ]

    result.loc[
        string_idx
    ] = pd.to_datetime(
        optc_raw.loc[string_idx],
        errors="coerce",
        utc=True
    )

    return result


train_data["timestamp"] = (
    convert_timestamp_by_source(
        train_data
    )
)

test_data["timestamp"] = (
    convert_timestamp_by_source(
        test_data
    )
)


print("TRAIN")
print(
    train_data.groupby(
        "dataset_source"
    )["timestamp"].apply(
        lambda x: x.notna().sum()
    )
)

print("\nTEST")
print(
    test_data.groupby(
        "dataset_source"
    )["timestamp"].apply(
        lambda x: x.notna().sum()
    )
)

TRAIN
dataset_source
OpTC           860398
Windows-APT     65461
Name: timestamp, dtype: int64

TEST
dataset_source
Windows-APT    36550
Name: timestamp, dtype: int64


In [149]:
# Ensure numeric fields are numeric
for col in ["pid", "ppid", "tid"]:

    train_data[col] = pd.to_numeric(
        train_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")

    test_data[col] = pd.to_numeric(
        test_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")


# Standardise categorical fields
for col in [
    "event_action",
    "event_object",
    "protocol"
]:

    train_data[col] = (
        train_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

    test_data[col] = (
        test_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )



### Feature Engineering

In [150]:
# I aggregate raw events into one-minute intervals/chunks
# For each chunk, the model receives total event count, unique process/thread counts,
# hour and minute, counts of the most common event actions, objects and protocols.

# Temporal settings
TIME_BIN = "1min"
LOOKBACK_MINUTES = 10
PREDICT_AHEAD_MINUTES = 3

TOP_ACTIONS = 30
TOP_OBJECTS = 20
TOP_PROTOCOLS = 10


# Learn categorical vocabulary from training data only
top_actions = (
    train_data["event_action"]
    .value_counts()
    .head(TOP_ACTIONS)
    .index
    .tolist()
)

top_objects = (
    train_data["event_object"]
    .value_counts()
    .head(TOP_OBJECTS)
    .index
    .tolist()
)

top_protocols = (
    train_data["protocol"]
    .value_counts()
    .head(TOP_PROTOCOLS)
    .index
    .tolist()
)


def clean_feature_name(value):
    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "_")
    )


action_columns = {
    value: f"action__{clean_feature_name(value)}"
    for value in top_actions
}

object_columns = {
    value: f"object__{clean_feature_name(value)}"
    for value in top_objects
}

protocol_columns = {
    value: f"protocol__{clean_feature_name(value)}"
    for value in top_protocols
}

In [151]:
def build_minute_features(data):

    df = data.copy()

    df["date"] = df["timestamp"].dt.date
    df["time_bin"] = df["timestamp"].dt.floor(TIME_BIN)

    # Base numerical features
    minute_data = (
        df.groupby(
            [
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            sort=False
        )
        .agg(
            event_count=("label", "size"),
            unique_pid=("pid", "nunique"),
            unique_ppid=("ppid", "nunique"),
            unique_tid=("tid", "nunique"),
            attack_now=("label", "max")
        )
        .reset_index()
    )

    # Count selected event actions
    action_data = df[
        df["event_action"].isin(top_actions)
    ].copy()

    if len(action_data) > 0:

        action_counts = (
            action_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "event_action"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=action_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            action_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    # Count selected event objects
    object_data = df[
        df["event_object"].isin(top_objects)
    ].copy()

    if len(object_data) > 0:

        object_counts = (
            object_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "event_object"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=object_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            object_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    # Count selected protocols
    protocol_data = df[
        df["protocol"].isin(top_protocols)
    ].copy()

    if len(protocol_data) > 0:

        protocol_counts = (
            protocol_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "protocol"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=protocol_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            protocol_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    minute_data = minute_data.fillna(0)

    # Keep the same time features used previously
    minute_data["hour"] = (
        minute_data["time_bin"]
        .dt.hour
        .astype("float32")
    )

    minute_data["minute"] = (
        minute_data["time_bin"]
        .dt.minute
        .astype("float32")
    )

    return minute_data


train_minutes = build_minute_features(
    train_data
)

test_minutes = build_minute_features(
    test_data
)





### Create Target Feature

In [152]:
def add_future_target(data):

    output = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    for _, group in data.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        ).copy()

        future_labels = []

        for step in range(
            1,
            PREDICT_AHEAD_MINUTES + 1
        ):

            future_labels.append(
                group["attack_now"]
                .shift(-step)
                .fillna(0)
                .to_numpy()
            )

        group["future_attack"] = (
            np.max(
                np.vstack(future_labels),
                axis=0
            ) > 0
        ).astype("int8")

        output.append(group)

    result = pd.concat(
        output,
        ignore_index=True
    )

    # Exclude minutes where the attack has already started
    result = result[
        result["attack_now"] == 0
    ].reset_index(drop=True)

    return result


train_minutes = add_future_target(
    train_minutes
)

test_minutes = add_future_target(
    test_minutes
)

print("Training target distribution:")
print(
    train_minutes["future_attack"]
    .value_counts()
)

print("\nTest target distribution:")
print(
    test_minutes["future_attack"]
    .value_counts()
)

Training target distribution:
future_attack
0    7424
1    2050
Name: count, dtype: int64

Test target distribution:
future_attack
1    490
0    435
Name: count, dtype: int64


In [153]:
## Clean DData for duplicates

train_minutes = build_minute_features(
    train_data
)

test_minutes = build_minute_features(
    test_data
)

# Remove duplicate columns after feature engineering
print("Train duplicates:")
print(
    train_minutes.columns[
        train_minutes.columns.duplicated()
    ].tolist()
)

print("\nTest duplicates:")
print(
    test_minutes.columns[
        test_minutes.columns.duplicated()
    ].tolist()
)

train_minutes = train_minutes.loc[
    :,
    ~train_minutes.columns.duplicated()
].copy()

test_minutes = test_minutes.loc[
    :,
    ~test_minutes.columns.duplicated()
].copy()

print(
    "\nTrain columns:",
    len(train_minutes.columns)
)

print(
    "Test columns:",
    len(test_minutes.columns)
)
train_minutes = add_future_target(train_minutes)
test_minutes = add_future_target(test_minutes)

Train duplicates:
['object__', 'protocol__']

Test duplicates:
['object__', 'protocol__']

Train columns: 59
Test columns: 27


### Scale and Encode

In [154]:
# Columns that should NOT be model features
metadata_columns = [
    "dataset_source",
    "hostname",
    "date",
    "time_bin",
    "attack_now",
    "future_attack"
]

# Define features from training data only
feature_columns = [
    col
    for col in train_minutes.columns
    if col not in metadata_columns
]

# Add any missing training columns to test
for col in feature_columns:
    if col not in test_minutes.columns:
        test_minutes[col] = 0

# Keep only the exact same feature columns
X_train_minutes = train_minutes[
    feature_columns
].copy()

X_test_minutes = test_minutes[
    feature_columns
].copy()

# Make everything numeric
X_train_minutes = X_train_minutes.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0).astype("float32")

X_test_minutes = X_test_minutes.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0).astype("float32")

print("Number of temporal features:", len(feature_columns))
print("Train feature shape:", X_train_minutes.shape)
print("Test feature shape:", X_test_minutes.shape)

assert X_train_minutes.shape[1] == len(feature_columns)
assert X_test_minutes.shape[1] == len(feature_columns)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_minutes
).astype("float32")

X_test_scaled = scaler.transform(
    X_test_minutes
).astype("float32")

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Number of temporal features: 54
Train feature shape: (9474, 54)
Test feature shape: (925, 54)
Scaled train shape: (9474, 54)
Scaled test shape: (925, 54)


### Create Temporal Sequences

In [155]:
def create_sequences(
    data,
    scaled_features,
    lookback=LOOKBACK_MINUTES
):

    X_sequences = []
    y_sequences = []
    metadata = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    # Keep row positions aligned with the scaled matrix
    working = data.reset_index(
        drop=True
    ).copy()

    working["_row_position"] = np.arange(
        len(working)
    )

    for _, group in working.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        )

        positions = group[
            "_row_position"
        ].to_numpy()

        labels = group[
            "future_attack"
        ].to_numpy()

        times = group[
            "time_bin"
        ].to_numpy()

        sources = group[
            "dataset_source"
        ].to_numpy()

        hosts = group[
            "hostname"
        ].to_numpy()

        for end_idx in range(
            lookback - 1,
            len(group)
        ):

            start_idx = (
                end_idx - lookback + 1
            )

            window_positions = positions[
                start_idx:end_idx + 1
            ]

            X_sequences.append(
                scaled_features[
                    window_positions
                ]
            )

            y_sequences.append(
                labels[end_idx]
            )

            metadata.append({
                "dataset_source":
                    sources[end_idx],
                "hostname":
                    hosts[end_idx],
                "window_end":
                    times[end_idx]
            })

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.int8
        ),
        pd.DataFrame(metadata)
    )


X_train_seq, Y_train_seq, train_seq_meta = (
    create_sequences(
        train_minutes,
        X_train_scaled
    )
)

X_test_seq, Y_test_seq, test_seq_meta = (
    create_sequences(
        test_minutes,
        X_test_scaled
    )
)


print(
    "LSTM/GRU train shape:",
    X_train_seq.shape
)

print(
    "LSTM/GRU test shape:",
    X_test_seq.shape
)

print("\nTraining sequence labels:")
print(
    pd.Series(Y_train_seq)
    .value_counts()
)

LSTM/GRU train shape: (8665, 10, 54)
LSTM/GRU test shape: (544, 10, 54)

Training sequence labels:
0    7031
1    1634
Name: count, dtype: int64


### Compute Class Weights

In [156]:
classes = np.unique(Y_train_seq)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=Y_train_seq
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print("Class weights:")
print(class_weights)

Class weights:
{0: 0.6161996870999857, 1: 2.651468788249694}


### Train ML Models

In [157]:
# Flatten temporal sequences for classical ML models
X_train_temporal = X_train_seq.reshape(
    X_train_seq.shape[0],
    -1
)


# Logistic Regression
lgr_model = LogisticRegression(
    max_iter=1000,
    random_state=7,
    class_weight="balanced"
)

lgr_model.fit(
    X_train_temporal,
    Y_train_seq
)

lgr_eval_pred = lgr_model.predict(
    X_train_temporal
)

print("===== Logistic Regression - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        lgr_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        lgr_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        lgr_eval_pred,
        digits=4,
        zero_division=0
    )
)


# XGBoost
xgb_model = XGBClassifier(
    random_state=7,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train_temporal,
    Y_train_seq
)

xgb_eval_pred = xgb_model.predict(
    X_train_temporal
)

print("\n===== XGBoost - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        xgb_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        xgb_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        xgb_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== Logistic Regression - Evaluation =====
Accuracy: 0.7791113675706867
Confusion Matrix:
 [[5158 1873]
 [  41 1593]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9921    0.7336    0.8435      7031
           1     0.4596    0.9749    0.6247      1634

    accuracy                         0.7791      8665
   macro avg     0.7259    0.8543    0.7341      8665
weighted avg     0.8917    0.7791    0.8022      8665


===== XGBoost - Evaluation =====
Accuracy: 0.9534910559723023
Confusion Matrix:
 [[6780  251]
 [ 152 1482]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9781    0.9643    0.9711      7031
           1     0.8552    0.9070    0.8803      1634

    accuracy                         0.9535      8665
   macro avg     0.9166    0.9356    0.9257      8665
weighted avg     0.9549    0.9535    0.9540      8665



### ML Validation

In [158]:
# Flatten test temporal sequences for classical ML models
X_test_temporal = X_test_seq.reshape(
    X_test_seq.shape[0],
    -1
)


# Logistic Regression Validation
lgr_val_pred = lgr_model.predict(
    X_test_temporal
)

print("===== Logistic Regression - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        lgr_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        lgr_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        lgr_val_pred,
        digits=4,
        zero_division=0
    )
)


# XGBoost Validation
xgb_val_pred = xgb_model.predict(
    X_test_temporal
)

print("\n===== XGBoost - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        xgb_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        xgb_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        xgb_val_pred,
        digits=4,
        zero_division=0
    )
)

===== Logistic Regression - Validation =====
Accuracy: 0.6139705882352942
Confusion Matrix:
 [[100 199]
 [ 11 234]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9009    0.3344    0.4878       299
           1     0.5404    0.9551    0.6903       245

    accuracy                         0.6140       544
   macro avg     0.7207    0.6448    0.5890       544
weighted avg     0.7386    0.6140    0.5790       544


===== XGBoost - Validation =====
Accuracy: 0.65625
Confusion Matrix:
 [[228  71]
 [116 129]]
Classification Report:
               precision    recall  f1-score   support

           0     0.6628    0.7625    0.7092       299
           1     0.6450    0.5265    0.5798       245

    accuracy                         0.6562       544
   macro avg     0.6539    0.6445    0.6445       544
weighted avg     0.6548    0.6562    0.6509       544



### Train and Evaluate LSTM

In [159]:
# LSTM Model

lstm_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),
    LSTM(
        128,
        return_sequences=True
    ),
    BatchNormalization(),
    Dropout(0.25),

    LSTM(64),
    BatchNormalization(),
    Dropout(0.25),

    Dense(
        64,
        activation="relu"
    ),
    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

lstm_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

lstm_history = lstm_model.fit(
    X_train_seq,
    Y_train_seq,
    validation_data=(
        X_test_seq,
        Y_test_seq
    ),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=lstm_callbacks,
    shuffle=False,
    verbose=1
)

Epoch 1/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.7638 - auc: 0.7561 - loss: 0.7428 - precision: 0.3975 - recall: 0.4902 - val_accuracy: 0.4504 - val_auc: 0.3951 - val_loss: 0.7831 - val_precision: 0.4504 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.7019 - auc: 0.7803 - loss: 0.5655 - precision: 0.3589 - recall: 0.7387 - val_accuracy: 0.4504 - val_auc: 0.6712 - val_loss: 0.8156 - val_precision: 0.4504 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7551 - auc: 0.8469 - loss: 0.4719 - precision: 0.4252 - recall: 0.8494 - val_accuracy: 0.4522 - val_auc: 0.8424 - val_loss: 0.7856 - val_precision: 0.4512 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 4/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.7745 - auc: 0.8678 - loss: 0.4268 - precision: 0.4500 - recall: 0.8819 - val_accuracy: 0.4890 - val_auc: 0.7199 - val_lo

In [160]:
# LSTM Evaluation on Training Data

lstm_eval_prob = lstm_model.predict(
    X_train_seq,
    verbose=0
).ravel()

lstm_eval_pred = (
    lstm_eval_prob >= 0.5
).astype(int)

print("===== LSTM - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        lstm_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        lstm_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        lstm_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== LSTM - Evaluation =====
Accuracy: 0.6924408540103866
Confusion Matrix:
 [[4400 2631]
 [  34 1600]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9923    0.6258    0.7676      7031
           1     0.3782    0.9792    0.5456      1634

    accuracy                         0.6924      8665
   macro avg     0.6852    0.8025    0.6566      8665
weighted avg     0.8765    0.6924    0.7257      8665



### LSTM Validation

In [161]:
# LSTM Validation on Test Data

lstm_val_prob = lstm_model.predict(
    X_test_seq,
    verbose=0
).ravel()

lstm_val_pred = (
    lstm_val_prob >= 0.5
).astype(int)

print("===== LSTM - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        lstm_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        lstm_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        lstm_val_pred,
        digits=4,
        zero_division=0
    )
)

===== LSTM - Validation =====
Accuracy: 0.45036764705882354
Confusion Matrix:
 [[  0 299]
 [  0 245]]
Classification Report:
               precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       299
           1     0.4504    1.0000    0.6210       245

    accuracy                         0.4504       544
   macro avg     0.2252    0.5000    0.3105       544
weighted avg     0.2028    0.4504    0.2797       544



### Train and Evaluate GRU

In [162]:
# GRU Model

gru_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),
    GRU(
        128,
        return_sequences=True
    ),
    BatchNormalization(),
    Dropout(0.25),

    GRU(64),
    BatchNormalization(),
    Dropout(0.25),

    Dense(
        64,
        activation="relu"
    ),
    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

gru_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

gru_history = gru_model.fit(
    X_train_seq,
    Y_train_seq,
    validation_data=(
        X_test_seq,
        Y_test_seq
    ),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=gru_callbacks,
    shuffle=False,
    verbose=1
)

Epoch 1/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.7234 - auc: 0.7236 - loss: 0.7226 - precision: 0.3501 - recall: 0.5453 - val_accuracy: 0.4504 - val_auc: 0.8234 - val_loss: 0.7798 - val_precision: 0.4504 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.6965 - auc: 0.7867 - loss: 0.5525 - precision: 0.3577 - recall: 0.7662 - val_accuracy: 0.4504 - val_auc: 0.7666 - val_loss: 0.9273 - val_precision: 0.4504 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.7452 - auc: 0.8434 - loss: 0.4762 - precision: 0.4140 - recall: 0.8458 - val_accuracy: 0.4926 - val_auc: 0.8566 - val_loss: 0.9114 - val_precision: 0.4702 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 4/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.7713 - auc: 0.8658 - loss: 0.4331 - precision: 0.4464 - recall: 0.8874 - val_accuracy: 0.5165 - val_auc: 0.8582 - val_lo

In [163]:
# GRU Evaluation on Training Data

gru_eval_prob = gru_model.predict(
    X_train_seq,
    verbose=0
).ravel()

gru_eval_pred = (
    gru_eval_prob >= 0.5
).astype(int)

print("===== GRU - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        gru_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        gru_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        gru_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== GRU - Evaluation =====
Accuracy: 0.7706866705135603
Confusion Matrix:
 [[5258 1773]
 [ 214 1420]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9609    0.7478    0.8411      7031
           1     0.4447    0.8690    0.5884      1634

    accuracy                         0.7707      8665
   macro avg     0.7028    0.8084    0.7147      8665
weighted avg     0.8636    0.7707    0.7934      8665



### GRU Validation

In [164]:
# GRU Validation - Test Data

gru_val_prob = gru_model.predict(
    X_test_seq,
    verbose=0
).ravel()

gru_val_pred = (
    gru_val_prob >= 0.5
).astype(int)

print("===== GRU - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        gru_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        gru_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        gru_val_pred,
        digits=4,
        zero_division=0
    )
)

===== GRU - Validation =====
Accuracy: 0.6525735294117647
Confusion Matrix:
 [[128 171]
 [ 18 227]]
Classification Report:
               precision    recall  f1-score   support

           0     0.8767    0.4281    0.5753       299
           1     0.5704    0.9265    0.7061       245

    accuracy                         0.6526       544
   macro avg     0.7235    0.6773    0.6407       544
weighted avg     0.7387    0.6526    0.6342       544



### Results Presentation

In [165]:
# TRAIN RESULTS

train_results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            lstm_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "GRU",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            gru_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "Logistic Regression",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            lgr_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "XGBoost",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            xgb_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        )
    }
])

print("TRAIN RESULTS")
display(train_results.round(4))

print(" ")


# TEST RESULTS

test_results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            lstm_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "GRU",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            gru_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "Logistic Regression",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            lgr_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "XGBoost",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            xgb_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        )
    }
])

print("TEST RESULTS")
display(test_results.round(4))

TRAIN RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,LSTM,0.6924,0.6852,0.8025,0.6566
1,GRU,0.7707,0.7028,0.8084,0.7147
2,Logistic Regression,0.7791,0.7259,0.8543,0.7341
3,XGBoost,0.9535,0.9166,0.9356,0.9257


 
TEST RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,LSTM,0.4504,0.2252,0.5000,0.3105
1,GRU,0.6526,0.7235,0.6773,0.6407
2,Logistic Regression,0.6140,0.7207,0.6448,0.5890
3,XGBoost,0.6562,0.6539,0.6445,0.6445
